<a href="https://colab.research.google.com/github/pravallikachejerla/Data-science/blob/main/dogcat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import os
import zipfile
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
import plotly.express as px
import random
from PIL import Image
import plotly.graph_objects as go
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/dogs-vs-cats/train.zip
/kaggle/input/dogs-vs-cats/test1.zip
/kaggle/input/dogs-vs-cats/sampleSubmission.csv


In [ ]:
import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil

CHUNK_SIZE = 40960
DATA_SOURCE_MAPPING = 'dogs-vs-cats:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-competitions-data%2Fkaggle-v2%2F3362%2F31148%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240813%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240813T054013Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D6910c8e8ec77ebbfef909bac683b97094bce99f7f9bec8de5cf9fc9c841d9acf6e8bb2af93aa6212ea11207aa4cbcbbaad44221efd753480ddcb66027c8eecabf021819bc8e384b6649c05d6bfe9337fd1953ef64e69079ddf9a3893688c005631fb39c26b65ac616063172425bb0b55ac544d708f9106f85ed65335922b718529bed755dbc78725d45e97d36aa28b3d7892eafebe6ae187ee30be40748650ecb51881fc600bad89f2ce7cce1e337e5f8c2f2535284ca81bc9dc1370b4022dfe4ef583c479e46bffc809781a5ec4572dbaa9257311092da97e48386337281f2c0437257ca5135534e9fa7c1dddd570134bee2175992f4c13c46c3ca8bf9b95e3'

KAGGLE_INPUT_PATH='/kaggle/input'
KAGGLE_WORKING_PATH='/kaggle/working'
KAGGLE_SYMLINK='kaggle'

!umount /kaggle/input/ 2> /dev/null
shutil.rmtree('/kaggle/input', ignore_errors=True)
os.makedirs(KAGGLE_INPUT_PATH, 0o777, exist_ok=True)
os.makedirs(KAGGLE_WORKING_PATH, 0o777, exist_ok=True)

try:
  os.symlink(KAGGLE_INPUT_PATH, os.path.join("..", 'input'), target_is_directory=True)
except FileExistsError:
  pass
try:
  os.symlink(KAGGLE_WORKING_PATH, os.path.join("..", 'working'), target_is_directory=True)
except FileExistsError:
  pass

for data_source_mapping in DATA_SOURCE_MAPPING.split(','):
    directory, download_url_encoded = data_source_mapping.split(':')
    download_url = unquote(download_url_encoded)
    filename = urlparse(download_url).path
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as fileres, NamedTemporaryFile() as tfile:
            total_length = fileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes compressed')
            dl = 0
            data = fileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = fileres.read(CHUNK_SIZE)
            if filename.endswith('.zip'):
              with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
            else:
              with tarfile.open(tfile.name) as tarfile:
                tarfile.extractall(destination_path)
            print(f'\nDownloaded and uncompressed: {directory}')
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue

print('Data source import complete.')


[==================================================] 851576689 bytes downloaded
Downloaded and uncompressed: dogs-vs-cats
Data source import complete.


In [ ]:
import numpy as np
import pandas as pd
import os
import zipfile
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
import plotly.express as px
import random
from PIL import Image
import plotly.graph_objects as go
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/dogs-vs-cats/train.zip
/kaggle/input/dogs-vs-cats/test1.zip
/kaggle/input/dogs-vs-cats/sampleSubmission.csv


In [ ]:
with zipfile.ZipFile('/kaggle/input/dogs-vs-cats/train.zip', 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/train')
with zipfile.ZipFile('/kaggle/input/dogs-vs-cats/test1.zip', 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/test')

In [ ]:
train_dir = '/kaggle/working/train/train'
test_dir = '/kaggle/working/test/test1'

In [ ]:
import os
import shutil
from tqdm import tqdm

# Define paths
train_dir = '/kaggle/working/train'
cat_dir = os.path.join(train_dir, 'cat')
dog_dir = os.path.join(train_dir, 'dog')

# Create directories if they don't exist
os.makedirs(cat_dir, exist_ok=True)
os.makedirs(dog_dir, exist_ok=True)

# Move files into respective directories
for filename in tqdm(os.listdir(train_dir), desc="Organizing images"):
    src_path = os.path.join(train_dir, filename)
    # Skip directories
    if os.path.isdir(src_path):
        continue
    # Check if the file should go to cat or dog directory
    if 'cat' in filename:
        dst_path = os.path.join(cat_dir, filename)
    elif 'dog' in filename:
        dst_path = os.path.join(dog_dir, filename)
    else:
        continue
    # Move the file
    shutil.move(src_path, dst_path)


Organizing images: 100%|██████████| 3/3 [00:00<00:00, 3185.55it/s]


In [ ]:
import os

train_dir = '/kaggle/working/train'
print("Contents of train_dir:")
print(os.listdir(train_dir))
print("Contents of train_dir/cat:")
print(os.listdir(os.path.join(train_dir, 'cat')))
print("Contents of train_dir/dog:")
print(os.listdir(os.path.join(train_dir, 'dog')))


Contents of train_dir:
['cat', 'dog', 'train']
Contents of train_dir/cat:
[]
Contents of train_dir/dog:
[]


In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

def load_images_and_labels(directory, image_size):
    images = []
    labels = []

    for label, class_name in enumerate(['cat', 'dog']):
        class_dir = os.path.join(directory, class_name)
        print(f"Loading images from: {class_dir}")
        try:
            file_list = os.listdir(class_dir)
            if not file_list:
                print(f"No images found in {class_dir}")
        except Exception as e:
            print(f"Error accessing {class_dir}: {e}")
            continue

        for filename in tqdm(file_list, desc=f"Loading {class_name} images"):
            img_path = os.path.join(class_dir, filename)
            image = cv2.imread(img_path)
            if image is None:
                print(f"Failed to load image: {img_path}")
                continue
            try:
                image = cv2.resize(image, image_size)
                images.append(image)
                labels.append(label)
            except Exception as e:
                print(f"Error processing image {img_path}: {e}")

    images = np.array(images)
    labels = np.array(labels)
    print(f"Loaded {len(images)} images with shape {image_size}")

    return images, labels


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Update train_dir and test_dir
train_dir = '/kaggle/working/train/train'
test_dir = '/kaggle/working/test/test1'

def load_images_and_labels(directory, image_size):
    images = []
    labels = []

    for label, class_name in enumerate(['cat', 'dog']):
        class_dir = os.path.join(directory, class_name)
        print(f"Loading images from: {class_dir}")
        try:
            file_list = os.listdir(class_dir)
            if not file_list:
                print(f"No images found in {class_dir}")
        except Exception as e:
            print(f"Error accessing {class_dir}: {e}")
            continue

        for filename in tqdm(file_list, desc=f"Loading {class_name} images"):
            img_path = os.path.join(class_dir, filename)
            image = cv2.imread(img_path)
            if image is None:
                print(f"Failed to load image: {img_path}")
                continue
            try:
                image = cv2.resize(image, image_size)
                images.append(image)
                labels.append(label)
            except Exception as e:
                print(f"Error processing image {img_path}: {e}")

    images = np.array(images)
    labels = np.array(labels)
    print(f"Loaded {len(images)} images with shape {image_size}")

    return images, labels

# Load training images with the correct path
X_train, y_train = load_images_and_labels(train_dir, (64, 64))

if X_train.size == 0:
    raise ValueError("No images were loaded. Please check the image loading process.")

# Continue with processing
X_train_flattened = X_train.reshape(X_train.shape[0], -1)

# Split data into training and validation sets
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train_flattened, y_train, test_size=0.2, random_state=42)

# Normalize the data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train the SVM model
from sklearn.svm import SVC

svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_scaled, y_train)

# Make predictions on validation data
y_pred = svm.predict(X_val_scaled)

# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:")
print(classification_report(y_val, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))


Loading images from: /kaggle/working/train/train/cat


Loading cat images: 100%|██████████| 1063/1063 [00:02<00:00, 356.52it/s]


Loading images from: /kaggle/working/train/train/dog


Loading dog images: 100%|██████████| 1023/1023 [00:02<00:00, 353.97it/s]


Loaded 2086 images with shape (64, 64)
Accuracy: 0.5598086124401914
Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.57      0.57       219
           1       0.54      0.55      0.54       199

    accuracy                           0.56       418
   macro avg       0.56      0.56      0.56       418
weighted avg       0.56      0.56      0.56       418

Confusion Matrix:
[[124  95]
 [ 89 110]]


In [ ]:
%%writefile image_classification.py

import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Update train_dir and test_dir
train_dir = '/kaggle/working/train/train'
test_dir = '/kaggle/working/test/test1'

def load_images_and_labels(directory, image_size):
    images = []
    labels = []

    for label, class_name in enumerate(['cat', 'dog']):
        class_dir = os.path.join(directory, class_name)
        print(f"Loading images from: {class_dir}")
        try:
            file_list = os.listdir(class_dir)
            if not file_list:
                print(f"No images found in {class_dir}")
        except Exception as e:
            print(f"Error accessing {class_dir}: {e}")
            continue

        for filename in tqdm(file_list, desc=f"Loading {class_name} images"):
            img_path = os.path.join(class_dir, filename)
            image = cv2.imread(img_path)
            if image is None:
                print(f"Failed to load image: {img_path}")
                continue
            try:
                image = cv2.resize(image, image_size)
                images.append(image)
                labels.append(label)
            except Exception as e:
                print(f"Error processing image {img_path}: {e}")

    images = np.array(images)
    labels = np.array(labels)
    print(f"Loaded {len(images)} images with shape {image_size}")

    return images, labels

# Load training images with the correct path
X_train, y_train = load_images_and_labels(train_dir, (64, 64))

if X_train.size == 0:
    raise ValueError("No images were loaded. Please check the image loading process.")

# Continue with processing
X_train_flattened = X_train.reshape(X_train.shape[0], -1)

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_flattened, y_train, test_size=0.2, random_state=42)

# Normalize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train the SVM model
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_scaled, y_train)

# Make predictions on validation data
y_pred = svm.predict(X_val_scaled)

# Evaluate the model
print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:")
print(classification_report(y_val, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))


Writing image_classification.py


In [ ]:
import pandas as pd

# Example: Assuming y_pred contains your predictions
submission_df = pd.DataFrame({
    'id': range(len(y_pred)),  # Replace with actual ID if necessary
    'prediction': y_pred
})

submission_df.to_csv('submission.csv', index=False)


In [ ]:
!pip install keras


In [ ]:
!pip install tensorflow


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def augment_images(images, labels):
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    augmented_images = []
    augmented_labels = []

    for image, label in zip(images, labels):
        image = np.expand_dims(image, 0)  # Expand dimensions to fit the datagen input
        it = datagen.flow(image, batch_size=1)
        for _ in range(5):  # Generate 5 augmented versions of each image
            aug_image = next(it)[0].astype(np.uint8)
            augmented_images.append(aug_image)
            augmented_labels.append(label)

    return np.array(augmented_images), np.array(augmented_labels)
